# Climate-Health Modeling Runner

Configure the network/mode/target here, then run the pipeline scripts.

In [ ]:
# ============================================================================
# CONFIGURATION - All parameters for climate-health modeling pipeline
# ============================================================================
from pathlib import Path
import subprocess
import sys

# --- Core Settings ---
NETWORK = 'NCD'                        # Network type: INF, NCD, SYNINF, SYNNCD
HSA_MODE = 'footprint'                 # HSA optimization mode: fewest, footprint, distance, etc.
TARGET_COL = 'diarrheal_count_adjusted' if NETWORK in ('INF', 'SYNINF') else 'hypertension_count_adjusted'

# --- Reproducibility ---
RANDOM_SEED = 42                       # Random seed for reproducibility

# --- Paths ---
OUT_DIR = Path('out/modeling')
INPUT_CSV = OUT_DIR / f'{NETWORK}_{HSA_MODE}_modeling_dataset.csv'
OUTPUT_PREFIX = f'{NETWORK}_{HSA_MODE}'

COMPREHENSIVE_DIR = OUT_DIR / 'results_comprehensive'
PARSIMONIOUS_DIR = OUT_DIR / 'results_parsimonious'
ML_RESULTS_DIR = OUT_DIR / 'results_ml'
IMPROVED_RESULTS_DIR = OUT_DIR / 'results_improved'
ANOMALIES_DIR = OUT_DIR / 'results_anomalies'

for d in [COMPREHENSIVE_DIR, PARSIMONIOUS_DIR, ML_RESULTS_DIR, IMPROVED_RESULTS_DIR, ANOMALIES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("="*80)
print("CLIMATE-HEALTH MODELING - CONFIGURATION")
print("="*80)
print(f'Network: {NETWORK}')
print(f'HSA mode: {HSA_MODE}')
print(f'Target column: {TARGET_COL}')
print(f'Random seed: {RANDOM_SEED}')
print(f'Input CSV: {INPUT_CSV}')
print(f'Output prefix: {OUTPUT_PREFIX}')
print("="*80)

In [ ]:
# Optional: re-run data prep + EDA plots
RUN_PREP = False

if RUN_PREP:
    cmd = [
        sys.executable, 'climate_health_modeling.py',
        '--network', NETWORK,
        '--hsa-mode', HSA_MODE,
        '--input-csv', str(INPUT_CSV),
        '--target-col', TARGET_COL,
        '--output-dir', str(OUT_DIR),
        '--output-prefix', OUTPUT_PREFIX,
    ]
    print('Running:', ' '.join(cmd))
    result = subprocess.run(cmd, check=False)
    if result.returncode != 0:
        raise RuntimeError(f'climate_health_modeling.py failed with code {result.returncode}')


In [ ]:
# Comprehensive modeling (variable pruning + model comparison)
cmd = [
    sys.executable, 'climate_health_modeling_comprehensive.py',
    '--network', NETWORK,
    '--hsa-mode', HSA_MODE,
    '--target-col', TARGET_COL,
    '--input-csv', str(INPUT_CSV),
    '--output-dir', str(COMPREHENSIVE_DIR),
    '--output-prefix', OUTPUT_PREFIX,
    '--random-seed', str(RANDOM_SEED),
]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, check=False)
if result.returncode != 0:
    raise RuntimeError(f'climate_health_modeling_comprehensive.py failed with code {result.returncode}')

In [ ]:
# Parsimonious modeling (theory-driven small models)
cmd = [
    sys.executable, 'climate_health_modeling_parsimonious.py',
    '--network', NETWORK,
    '--hsa-mode', HSA_MODE,
    '--target-col', TARGET_COL,
    '--input-csv', str(INPUT_CSV),
    '--output-dir', str(PARSIMONIOUS_DIR),
    '--output-prefix', OUTPUT_PREFIX,
    '--random-seed', str(RANDOM_SEED),
]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, check=False)
if result.returncode != 0:
    raise RuntimeError(f'climate_health_modeling_parsimonious.py failed with code {result.returncode}')

In [ ]:
# ML model training (baseline + multiple ML families)
cmd = [
    sys.executable, 'train_ml_models.py',
    '--network', NETWORK,
    '--hsa-mode', HSA_MODE,
    '--target-col', TARGET_COL,
    '--data-dir', str(OUT_DIR),
    '--output-dir', str(ML_RESULTS_DIR),
    '--random-seed', str(RANDOM_SEED),
]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, check=False)
if result.returncode != 0:
    raise RuntimeError(f'train_ml_models.py failed with code {result.returncode}')

In [ ]:
# Improved ML models (alternative feature sets + comparisons)
cmd = [
    sys.executable, 'train_improved_models.py',
    '--network', NETWORK,
    '--hsa-mode', HSA_MODE,
    '--target-col', TARGET_COL,
    '--data-dir', str(OUT_DIR),
    '--output-dir', str(IMPROVED_RESULTS_DIR),
    '--random-seed', str(RANDOM_SEED),
]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, check=False)
if result.returncode != 0:
    raise RuntimeError(f'train_improved_models.py failed with code {result.returncode}')

In [ ]:
# Anomaly-based modeling (tests if climate anomalies predict disease anomalies)
# This removes the shared seasonal signal and tests for direct climate effects
cmd = [
    sys.executable, 'climate_health_modeling_anomalies.py',
    '--network', NETWORK,
    '--hsa-mode', HSA_MODE,
    '--target-col', TARGET_COL,
    '--input-csv', str(INPUT_CSV),
    '--output-dir', str(ANOMALIES_DIR),
    '--output-prefix', OUTPUT_PREFIX,
    '--random-seed', str(RANDOM_SEED),
]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, check=False)
if result.returncode != 0:
    raise RuntimeError(f'climate_health_modeling_anomalies.py failed with code {result.returncode}')